In [2]:
from typing import Optional
from pathlib import Path
import json
import datetime
from urllib.parse import urlparse
import pandas as pd


In [3]:
def get_data(data_url: str) -> pd.DataFrame:
    """
    Function to download data from a public URL and return it as a pandas DataFrame.
    """
    # Read the CSV file from the URL
    df = pd.read_csv(data_url)
    return df

def extract_url_components(url: str) -> tuple[str, str]:
    
    domain = urlparse(url).netloc
    share_id = urlparse(url).path.split('/')[-1]
    
    return domain, share_id

def build_file_url(
    file_id: str, 
    ext: str, 
    url: Optional[str] = None, 
    domain: Optional[str] = None, 
    share_id: Optional[str] = None
) -> str:
    """Builds a URL for accessing a file on a webdav server.
    If a full URL is provided, it extracts the domain and share_id from it.
    If only domain and share_id are provided, it constructs the URL accordingly.
    If share_id is None, it assumes the file is being uploaded to webdav.
    """
    
    assert (url is not None) or (domain is not None), \
        "Either a full URL or both domain and share_id must be provided."
    if url is not None:
        domain, share_id = extract_url_components(url)
        
    if share_id is None: # When uploading to webdav
        return f"https://{domain}/public.php/webdav/{file_id}.{ext}"
    else: # When downloading from webdav
        return f"https://{domain}/public.php/dav/files/{share_id}/{file_id}.{ext}"


url: str = "https://collab.psa.es/s/WR6MxyJsnZWi9xH"
env_file_id: str = "environment"

request_url = build_file_url(            
    url=url,
    file_id=env_file_id,
    ext="csv"
)
get_data(request_url)


,Unnamed: 0,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,...,0,Q_kW,Tv_C,mv_kgh,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_m3,Vavail_l
0,2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,...,NaN,200.0,35.0,297.774165,0.029054,9.1920,0.000029,0.009192,0.886583,886.583184
1,2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,...,NaN,200.0,35.0,297.774165,0.029054,9.1096,0.000029,0.009110,0.886583,886.583184
2,2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,...,NaN,200.0,35.0,297.774165,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
3,2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,...,NaN,200.0,35.0,297.774165,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
4,2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,...,NaN,200.0,35.0,297.774165,0.029054,7.6592,0.000029,0.007659,0.886583,886.583184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26299,2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,...,NaN,200.0,35.0,297.774165,0.029054,12.4880,0.000029,0.012488,0.991771,991.771020
26300,2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,...,NaN,200.0,35.0,297.774165,0.029054,12.0680,0.000029,0.012068,0.991771,991.771020
26301,2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,...,NaN,200.0,35.0,297.774165,0.029054,11.5520,0.000029,0.011552,0.991771,991.771020
26302,2024-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,...,NaN,200.0,35.0,297.774165,0.029054,11.1496,0.000029,0.011150,0.991771,991.771020


In [ ]:
from pathlib import Path
import pandas as pd
import datetime
from solhycool_optimization import DayResults

def create_mock_day_results(date_str: str, template_path: Path, template_date_str: str = "20220501") -> DayResults:
    """
    Create a new DayResults object with the same content as the template,
    but with timestamps shifted to match the given date_str.
    """
    # 1. Load from existing file
    original = DayResults.initialize(template_path, date_str=template_date_str)

    # 2. Convert new date_str to datetime
    new_date = pd.to_datetime(date_str, format="%Y%m%d")
    old_date = pd.to_datetime(template_date_str, format="%Y%m%d")
    delta_days = (new_date - old_date).days

    # 3. Shift index and time-dependent fields
    new_index = original.index + pd.Timedelta(days=delta_days)
    new_df_results = original.df_results.copy()
    new_df_results.index = new_df_results.index + pd.Timedelta(days=delta_days)

    # Optional: shift fitness history index if needed (not always datetime-indexed)
    fitness_history = original.fitness_history.copy()
    if isinstance(fitness_history.index, pd.DatetimeIndex):
        fitness_history.index = fitness_history.index + pd.Timedelta(days=delta_days)

    return DayResults(
        index=new_index,
        df_paretos=original.df_paretos,  # usually per-timestep, no timestamp inside
        fitness_history=fitness_history,
        selected_pareto_idxs=original.selected_pareto_idxs,
        df_results=new_df_results,
        consumption_arrays=original.consumption_arrays,
        pareto_idxs=original.pareto_idxs,
        date_str=date_str
    )

# Use current date
mock_date_str = datetime.datetime.today().strftime("%Y%m%d")

mock_day_results = create_mock_day_results(
    date_str=mock_date_str,
    template_path=Path("../dags/results_eval_at_20250421T1741_psa_partial.gz"),
    template_date_str="20220501"
)

# Now you can export or use mock_day_results downstream
mock_day_results.export(Path(f"/tmp/mock_day_results_{mock_date_str}.h5"))


2025-07-21 08:13:30.873 | INFO     | solhycool_optimization:initialize:201 - DayResults loaded for 20220501 from ../dags/results_eval_at_20250421T1741_psa_partial.gz
2025-07-21 08:13:31.402 | INFO     | solhycool_optimization:export:267 - Results for 20250721 saved to /tmp/mock_day_results_20250721.h5


In [6]:
mock_day_results.index[0].dt


AttributeError: 'Timestamp' object has no attribute 'dt'

In [1]:
from pathlib import Path
import hjson
from solhycool_optimization import DayResults
from solhycool_visualization.objects import HorizonResultsVisualizer

plot_config = hjson.loads(Path("../../data/plot_config_day_horizon.hjson").read_text())
day_results = DayResults.initialize(Path("../dags/results_eval_at_20250421T1741_psa_partial.gz"), date_str="20220501")

visualizer = HorizonResultsVisualizer(
    results_plot_config=plot_config,
    day_results=day_results,
)

visualizer.generate_all(
    output_path=Path("."),
    formats=["png"]
)


2025-07-21 12:55:10.112 | INFO     | solhycool_optimization:initialize:201 - DayResults loaded for 20220501 from ../dags/results_eval_at_20250421T1741_psa_partial.gz
2025-07-21 12:55:11.042 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_0.png
2025-07-21 12:55:11.259 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_1.png
2025-07-21 12:55:11.530 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_2.png
2025-07-21 12:55:11.582 | INFO     | phd_visualizations:save_figure:41 - Figure saved in fitness_history_path_selection.png
2025-07-21 12:55:12.704 | INFO     | phd_visualizations:save_figure:41 - Figure saved in results.png
2025-07-21 12:55:12.705 | INFO     | solhycool_visualization.objects:generate_all:128 - Visualizations saved to .
